# 03. 고성능 모델

- 담당: **③ 안주형 (고성능 모델)** (이 파일은 담당자만 수정)
- 기록 파일: `experiments/log_A_model.csv`

### 할 일
- [ ] XGBoost 개선, LightGBM·CatBoost 비교 (같은 fold, 같은 피처)
- [ ] 클래스 불균형 대응 (`balanced_weights` 등)
- [ ] 튜닝, 앙상블 후보 (cv['oof_proba'] 확률 평균)
- [ ] seed·하이퍼파라미터를 코드에 숫자로 남기기

> **규칙:** 전처리는 Pipeline 안에 / test는 보지 않기 / 점수는 `run_cv()`로 / 결과는 `log_result()`로 기록 / 자세한 규칙은 `docs/coding_rules.md`

## 0. 환경 설정 (수정하지 말고 그대로 실행)
Colab은 구글 드라이브 `내 드라이브`에 레포 폴더(`ah07-cancer-subtype`, 안에 `data/` 포함)를 두면 돼요.

In [8]:
import os, sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = "/content/drive/MyDrive/ah07-cancer-subtype"  # 드라이브에 둔 레포 폴더
else:
    REPO_DIR = os.path.abspath(os.path.join("..", ".."))  # notebooks/이니셜_역할/ 폴더 기준

# 경로 확인: 여기서 멈추면 폴더 위치나 이름을 확인해 주세요
assert os.path.isdir(os.path.join(REPO_DIR, "src")), (
    f"src 폴더를 찾을 수 없어요: {REPO_DIR}\n"
    "→ 드라이브 '내 드라이브' 바로 아래에 'ah07-cancer-subtype' 폴더가 있는지, "
    "ZIP을 풀 때 폴더가 두 번 겹치지 않았는지(ah07-cancer-subtype/ah07-cancer-subtype-dev) 확인")
assert os.path.isfile(os.path.join(REPO_DIR, "data", "train.csv")), (
    "data/train.csv 가 없어요 → 데이콘 csv 3개를 data/ 폴더에 넣어 주세요")
sys.path.insert(0, REPO_DIR)

from src.config import SEED, DATA_DIR, set_seed
set_seed(SEED)
print("REPO_DIR:", REPO_DIR, "| DATA_DIR:", DATA_DIR)

REPO_DIR: c:\Users\user\Desktop\AI_Projects\ah07-cancer-subtype | DATA_DIR: C:\Users\user\Desktop\AI_Projects\ah07-cancer-subtype\data


In [9]:
import platform
import numpy as np, pandas as pd, sklearn
from sklearn.pipeline import Pipeline

from src.data_loader import load_data, split_xy, encode_target
from src.preprocessing import basic_transformer, binarize_mutations
from src.features import mutation_count
from src.validation import get_folds, run_cv, balanced_weights
from src.evaluation import macro_f1, per_class_f1, log_result, save_predictions, load_predictions
from src.prediction import make_submission

print("OS:", platform.platform(), "| python", platform.python_version(),
      "| pandas", pd.__version__, "| scikit-learn", sklearn.__version__)
from sklearn.metrics import f1_score


OS: Windows-11-10.0.26200-SP0 | python 3.14.7 | pandas 3.1.1 | scikit-learn 1.9.1


### 실행 장치 자동 선택 (공통 `src.config.DEVICE` 연동)

XGBoost는 CUDA 사용 가능 시 GPU를 선택하며, 지원하지 않으면 CPU로 실행합니다. 팀 공통 설정은 박유정님과 합의 후 추가합니다.

In [10]:
# GPU 자동 선택 (GPU 없으면 CPU)
# 박유정님과 합의 전까지 src/config.py는 수정하지 않음

import src.config as project_config
from xgboost import XGBClassifier
import json
import warnings

REQUESTED_DEVICE = getattr(project_config, "DEVICE", "auto")

if REQUESTED_DEVICE not in ("auto", "cpu", "cuda"):
    raise ValueError("DEVICE는 auto, cpu, cuda 중 하나여야 합니다.")


def resolve_xgb_device(requested=REQUESTED_DEVICE):

    if requested == "cpu":
        return "cpu"

    try:
        probe_x = np.array(
            [[0., 1.], [1., 0.], [0.2, 0.8], [0.8, 0.2]],
            dtype=np.float32
        )
        probe_y = np.array([0, 1, 0, 1])

        probe = XGBClassifier(
            n_estimators=2,
            max_depth=1,
            tree_method="hist",
            device="cuda",
            n_jobs=1
        )

        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            probe.fit(probe_x, probe_y)

        config = json.loads(probe.get_booster().save_config())
        actual_device = config["learner"]["generic_param"]["device"]

        if actual_device.startswith("cuda"):
            return "cuda"

        raise RuntimeError(
            f"GPU 대신 {actual_device} 장치가 선택되었습니다."
        )

    except Exception as exc:
        if requested == "cuda":
            raise RuntimeError(
                "CUDA를 요청했지만 XGBoost GPU를 사용할 수 없습니다."
            ) from exc

        print("GPU 사용 불가 → CPU 자동 전환:", exc)
        return "cpu"


XGB_DEVICE = resolve_xgb_device()

print(f"설정 DEVICE={REQUESTED_DEVICE}")
print(f"실제 XGBoost 장치={XGB_DEVICE}")

설정 DEVICE=auto
실제 XGBoost 장치=cuda


## 1. 데이터

In [11]:
train, test = load_data(DATA_DIR)   # test는 형식 확인과 최종 추론에만
X, y_raw = split_xy(train)
y, le = encode_target(y_raw)
folds = get_folds(y, X=X, classes=le.classes_)                 # 팀 공통 fold
print("train", train.shape, "| test", test.shape, "| classes", len(le.classes_))

train (6201, 4386) | test (2546, 4385) | classes 26


> **환경 확인:** 위에서 `REPO_DIR`과 `os`를 정의한 후 경로를 출력합니다. 원본에서는 이 셀이 환경 설정보다 앞에 있어 처음부터 실행하면 오류가 발생할 수 있었습니다.

In [12]:
print("현재 작업 경로:", os.getcwd())
print("프로젝트 경로:", REPO_DIR)
print("데이터 폴더:", os.path.join(REPO_DIR, "data"))

현재 작업 경로: c:\Users\user\Desktop\AI_Projects\ah07-cancer-subtype\notebooks\A_model
프로젝트 경로: c:\Users\user\Desktop\AI_Projects\ah07-cancer-subtype
데이터 폴더: c:\Users\user\Desktop\AI_Projects\ah07-cancer-subtype\data


## 2. 실험 설계 및 데이터 확인

**주의:** 과거에 Markdown에 적힌 점수는 이전 Stratified K-Fold 실행 기록입니다. 현재 `folds`는 쌍둥이 Group K-Fold이므로 **재실행 결과로만 평가**하고, 기존 숫자를 새 검증 결과로 인용하지 마세요. 모든 모델은 공통 Fold를 사용합니다.


In [13]:
# 변이 데이터 품질 점검: 문자열을 임의의 실수로 변환하지 않음
print("X shape:", X.shape)
print("자료형 분포:")
display(X.dtypes.value_counts())
print("원본 결측값:", int(X.isna().sum().sum()))
print("WT 비율:", round(float(X.eq("WT").to_numpy().mean()), 5))
mutation_binary = binarize_mutations(X)
print("이진 변이 행렬:", mutation_binary.shape)
print("변이 보유 유전자 수(상위 10개):")
display(mutation_binary.sum().sort_values(ascending=False).head(10))
del mutation_binary

X shape: (6201, 4384)
자료형 분포:


str    4384
Name: count, dtype: int64

원본 결측값: 0
WT 비율: 0.99195
이진 변이 행렬: (6201, 4384)
변이 보유 유전자 수(상위 10개):


TP53      1770
PIK3CA     686
RYR2       643
SYNE1      642
PCLO       593
RYR1       473
SPTA1      461
KMT2D      451
IDH1       441
BRAF       438
dtype: int64

## 3. LightGBM 모델 실험

**목적:** 기본 모델(V1), 복잡도 변경(V2), 클래스 불균형 가중치(V3), 가중치 강도(V4)를 동일 Fold로 비교합니다. 소수 암종도 동등하게 반영하기 위해 Macro F1을 사용합니다.

**LightGBM V1 모델 정의:** 팀 공통 이진 변이 전처리와 LightGBM을 Pipeline으로 결합합니다. 모델 설정을 한 곳에서 관리합니다.

In [14]:
from lightgbm import LGBMClassifier

LGBM_PARAMS = {
    "n_estimators": 200,
    "learning_rate": 0.05,
    "num_leaves": 15,
    "max_depth": 6,
    "random_state": SEED,
    "verbosity": -1,
    "n_jobs": -1
}

lgbm_pipe = Pipeline([
    ("features", basic_transformer()),
    ("model", LGBMClassifier(**LGBM_PARAMS))
])

print("LightGBM 모델 생성 완료")

LightGBM 모델 생성 완료


**V1 검증 결과:** 5-Fold 평균 Macro F1 **0.3030**. 기준 성능으로 보관합니다.

In [15]:
# LightGBM 교차검증
lgbm_cv = run_cv(lgbm_pipe, X, y, folds=folds)

# 결과 확인
display(lgbm_cv)

fold 1: Macro F1 = 0.3297
fold 2: Macro F1 = 0.3324
fold 3: Macro F1 = 0.3118
fold 4: Macro F1 = 0.3283
fold 5: Macro F1 = 0.3379
CV Macro F1 = 0.3280 (std 0.0088)


{'scores': [0.3297469046067294,
  0.33243523774511363,
  0.3118070034081765,
  0.32825551330308855,
  0.33789643800345104],
 'mean': 0.32802821941331184,
 'std': 0.008751422041145847,
 'oof': array([12, 19, 20, ...,  4,  8, 20], shape=(6201,)),
 'oof_proba': array([[1.35506265e-04, 2.05216082e-03, 1.13330196e-01, ...,
         2.50921368e-02, 5.36892192e-02, 4.59492075e-02],
        [5.41493208e-05, 4.31194239e-02, 1.42174929e-01, ...,
         3.79580441e-03, 2.20929029e-03, 5.42560923e-03],
        [6.11980116e-06, 3.87431624e-05, 1.54986576e-04, ...,
         3.95780567e-05, 2.18120346e-06, 2.99864233e-05],
        ...,
        [3.22119279e-05, 7.69522329e-04, 2.34674011e-02, ...,
         7.28531368e-03, 2.69885626e-03, 1.60344128e-02],
        [1.04340465e-04, 6.49615029e-03, 1.02864688e-01, ...,
         9.49991764e-02, 6.47861070e-02, 1.71732362e-02],
        [2.52870411e-05, 4.52226241e-04, 3.82917922e-02, ...,
         1.30594963e-02, 1.38617175e-04, 1.84593070e-03]], shape=(6

**V2 실험:** 트리 복잡도와 반복 횟수를 높였습니다. 평균 Macro F1 **0.3004**로 V1보다 낮아 개선안으로 채택하지 않습니다.

In [16]:
from sklearn.base import clone

# 실험 2: LightGBM 하이퍼파라미터 변경
lgbm_v2 = clone(lgbm_pipe)

lgbm_v2.set_params(
    model__n_estimators=400,
    model__learning_rate=0.03,
    model__num_leaves=31,
    model__max_depth=8,
    model__min_child_samples=30,
    model__device_type="cpu"
)

# 기존과 동일한 교차검증
lgbm_cv_v2 = run_cv(lgbm_v2, X, y, folds=folds)

# 성능 비교
print(f"기존 모델 Macro F1: {lgbm_cv['mean']:.4f}")
print(f"개선 실험 Macro F1: {lgbm_cv_v2['mean']:.4f}")
print(f"성능 차이: {lgbm_cv_v2['mean'] - lgbm_cv['mean']:+.4f}")

fold 1: Macro F1 = 0.3289
fold 2: Macro F1 = 0.3221
fold 3: Macro F1 = 0.2967
fold 4: Macro F1 = 0.3174
fold 5: Macro F1 = 0.3361
CV Macro F1 = 0.3203 (std 0.0134)
기존 모델 Macro F1: 0.3280
개선 실험 Macro F1: 0.3203
성능 차이: -0.0078


**V1 클래스별 분석:** 클래스별 precision/recall/F1을 확인해 전체 정확도만으로 놓치기 쉬운 소수 클래스 문제를 살펴봅니다.

In [17]:
from sklearn.metrics import classification_report, confusion_matrix
class_names = np.asarray(le.classes_)
class_ids = np.arange(len(class_names))
print("=== LightGBM V1 전체 암종별 성능 (Train OOF) ===")
print(classification_report(y, lgbm_cv['oof'], labels=class_ids,
                            target_names=class_names, zero_division=0, digits=4))
display(pd.DataFrame({'암종':class_names,'실제 건수':np.bincount(y,minlength=len(class_names))}))


=== LightGBM V1 전체 암종별 성능 (Train OOF) ===
              precision    recall  f1-score   support

         ACC     0.9362    0.6111    0.7395        72
        BLCA     0.5588    0.1827    0.2754       104
        BRCA     0.2953    0.6387    0.4039       786
        CESC     0.1264    0.0710    0.0909       155
        COAD     0.6059    0.6413    0.6231       223
        DLBC     0.7273    0.2105    0.3265        38
      GBMLGG     0.4341    0.4143    0.4240       461
        HNSC     0.3089    0.2646    0.2850       223
       KIPAN     0.3175    0.3631    0.3388       515
        KIRC     0.3725    0.2275    0.2825       334
        LAML     0.5086    0.3734    0.4307       158
         LGG     0.4644    0.4847    0.4744       229
        LIHC     0.3077    0.2278    0.2618       158
        LUAD     0.2874    0.1359    0.1845       184
        LUSC     0.3836    0.1573    0.2231       178
          OV     0.2633    0.4308    0.3268       253
        PAAD     0.1000    0.0417    0.

,암종,실제 건수
0,ACC,72
1,BLCA,104
2,BRCA,786
3,CESC,155
4,COAD,223
5,DLBC,38
6,GBMLGG,461
7,HNSC,223
8,KIPAN,515
9,KIRC,334


**V3 실험:** LightGBM `class_weight="balanced"`를 사용합니다. 평균 Macro F1 **0.3286**으로 LightGBM 실험 중 가장 높았습니다.

In [18]:
# 실험 3: 클래스 불균형 보정

lgbm_v3 = clone(lgbm_pipe)

lgbm_v3.set_params(
    model__class_weight="balanced",
    model__device_type="cpu"
)

lgbm_cv_v3 = run_cv(
    lgbm_v3,
    X,
    y,
    folds=folds
)

print("\n=== LightGBM 실험 비교 ===")
print(f"V1 기본 모델: {lgbm_cv['mean']:.4f}")
print(f"V2 파라미터 변경: {lgbm_cv_v2['mean']:.4f}")
print(f"V3 클래스 가중치: {lgbm_cv_v3['mean']:.4f}")

fold 1: Macro F1 = 0.3430
fold 2: Macro F1 = 0.3560
fold 3: Macro F1 = 0.3328
fold 4: Macro F1 = 0.3325
fold 5: Macro F1 = 0.3742
CV Macro F1 = 0.3477 (std 0.0158)

=== LightGBM 실험 비교 ===
V1 기본 모델: 0.3280
V2 파라미터 변경: 0.3203
V3 클래스 가중치: 0.3477


**V1·V3 클래스별 비교:** 평균 개선이 모든 암종의 개선을 의미하지는 않으므로 클래스별 F1 차이를 기록합니다.

In [19]:
# V1과 V3의 전체 암종별 성능 비교 (숫자 라벨은 모델 내부에서만 사용)
report_v1 = classification_report(y,lgbm_cv['oof'],labels=class_ids,target_names=class_names,output_dict=True,zero_division=0)
report_v3 = classification_report(y,lgbm_cv_v3['oof'],labels=class_ids,target_names=class_names,output_dict=True,zero_division=0)
comparison = pd.DataFrame([{'암종':name,'데이터 수':report_v1[name]['support'],
    'V1 F1':report_v1[name]['f1-score'],'V3 F1':report_v3[name]['f1-score']}
    for name in class_names])
comparison['F1 변화'] = comparison['V3 F1']-comparison['V1 F1']
display(comparison.sort_values('F1 변화',ascending=False).round(4))
print('V3 전체 OOF Macro F1:',round(report_v3['macro avg']['f1-score'],4))


,암종,데이터 수,V1 F1,V3 F1,F1 변화
24,THYM,98.0,0.0000,0.1632,0.1632
9,KIRC,334.0,0.2825,0.4184,0.1359
11,LGG,229.0,0.4744,0.5611,0.0867
2,BRCA,786.0,0.4039,0.4854,0.0815
16,PAAD,120.0,0.0588,0.1367,0.0779
18,PRAD,266.0,0.1814,0.2312,0.0498
17,PCPG,147.0,0.1364,0.1836,0.0473
0,ACC,72.0,0.7395,0.7717,0.0322
14,LUSC,178.0,0.2231,0.2509,0.0278
15,OV,253.0,0.3268,0.3509,0.0240


V3 전체 OOF Macro F1: 0.3495


**V3 혼동 분석:** 특정 클래스의 과다 예측과 다른 클래스로의 혼동을 확인합니다.

In [20]:
# 전체 26개 암종에서 실제 -> 예측 오분류 조합 확인
cm_v3 = confusion_matrix(y,lgbm_cv_v3['oof'],labels=class_ids)
errors_v3 = [{'실제 암종':class_names[i],'예측 암종':class_names[j],
              '오분류 건수':int(cm_v3[i,j])}
             for i in class_ids for j in class_ids if i!=j and cm_v3[i,j]>0]
display(pd.DataFrame(errors_v3).sort_values('오분류 건수',ascending=False).head(30))


,실제 암종,예측 암종,오분류 건수
120,GBMLGG,LGG,177
163,KIPAN,KIRC,147
45,BRCA,OV,106
54,BRCA,THYM,106
362,PRAD,THYM,95
466,THCA,THYM,77
343,PCPG,THYM,61
176,KIPAN,THYM,46
209,LAML,THYM,46
171,KIPAN,PRAD,39


**예측 분포 비교:** 클래스별 실제 빈도와 V1·V3 예측 빈도를 비교합니다.

In [21]:
# Train OOF만 사용: 실제/예측 분포 비교. Test 예측 분포 분석 금지.
prediction_comparison = pd.DataFrame({
    '암종':class_names,
    '실제 데이터 수':np.bincount(y,minlength=len(class_names)),
    'V1 예측 수':np.bincount(lgbm_cv['oof'],minlength=len(class_names)),
    'V3 예측 수':np.bincount(lgbm_cv_v3['oof'],minlength=len(class_names))})
prediction_comparison['V3 예측 변화']=prediction_comparison['V3 예측 수']-prediction_comparison['V1 예측 수']
prediction_comparison['V3 예측/실제 비율']=(prediction_comparison['V3 예측 수']/prediction_comparison['실제 데이터 수'].replace(0,np.nan))
display(prediction_comparison.sort_values('V3 예측 변화',ascending=False).round(3))


,암종,실제 데이터 수,V1 예측 수,V3 예측 수,V3 예측 변화,V3 예측/실제 비율
24,THYM,98,23,674,651,6.878
17,PCPG,147,117,256,139,1.741
11,LGG,229,239,377,138,1.646
9,KIRC,334,204,340,136,1.018
18,PRAD,266,186,305,119,1.147
16,PAAD,120,50,158,108,1.317
19,SARC,198,123,220,97,1.111
3,CESC,155,87,168,81,1.084
22,TGCT,124,126,204,78,1.645
15,OV,253,414,488,74,1.929


**V4 실험:** `0.25 / 0.50 / 0.75`의 부분 가중치를 각 학습 Fold에서만 계산합니다. 결과는 모두 V3의 **0.3286**보다 낮았습니다. 가중치 중복 적용을 방지합니다.

In [22]:
# ============================================================
# V4: 클래스 가중치 강도별 LightGBM 실험
# - 팀 공통 run_cv() 사용
# - 기존 V1 모델의 하이퍼파라미터 유지
# - 각 Fold의 학습 데이터에서만 가중치 계산
# - 평가 지표: 5-Fold CV Macro F1
# ============================================================

from sklearn.utils.class_weight import compute_sample_weight

# 1. 학습 Fold 내부에서 가중치를 계산하는 함수
def make_weight_fn(strength):

    def weight_fn(y_tr):

        balanced = compute_sample_weight(
            class_weight="balanced",
            y=y_tr
        )

        # strength=0: 가중치 없음
        # strength=1: 완전한 balanced
        weights = 1.0 + strength * (balanced - 1.0)

        return {
            "model__sample_weight": weights
        }

    return weight_fn


# 2. 실험할 가중치 강도
weight_strengths = [0.25, 0.50, 0.75]

v4_results = {}
v4_summary = []

# 3. 동일한 모델, 동일한 Fold로 실험
for strength in weight_strengths:

    print("\n" + "=" * 60)
    print(f"V4 실험 | 가중치 강도: {strength}")
    print("=" * 60)

    model = clone(lgbm_pipe)

    # V1의 기본 모델을 사용하며 이중 가중치 적용 방지
    model.set_params(
        model__class_weight=None,
        model__device_type="cpu"
    )

    result = run_cv(
        model,
        X,
        y,
        folds=folds,
        fit_params_fn=make_weight_fn(strength)
    )

    v4_results[strength] = result

    v4_summary.append({
        "모델": f"V4_{strength}",
        "가중치 강도": strength,
        "CV Macro F1": result["mean"],
        "CV Std": result["std"]
    })


# 4. 기존 모델과 비교
comparison_results = pd.DataFrame([
    {
        "모델": "V1",
        "가중치 강도": 0.0,
        "CV Macro F1": lgbm_cv["mean"],
        "CV Std": lgbm_cv["std"]
    },
    {
        "모델": "V3",
        "가중치 강도": 1.0,
        "CV Macro F1": lgbm_cv_v3["mean"],
        "CV Std": lgbm_cv_v3["std"]
    },
    *v4_summary
])

comparison_results = comparison_results.sort_values(
    "CV Macro F1",
    ascending=False
).reset_index(drop=True)

print("\n=== 전체 가중치 실험 비교 ===")
display(comparison_results.round(4))


# 5. 최고 성능 모델 확인
best_row = comparison_results.iloc[0]

print("\n최고 성능 모델:", best_row["모델"])
print("가중치 강도:", best_row["가중치 강도"])
print("CV Macro F1:", round(best_row["CV Macro F1"], 4))


V4 실험 | 가중치 강도: 0.25
fold 1: Macro F1 = 0.3353
fold 2: Macro F1 = 0.3457
fold 3: Macro F1 = 0.3125
fold 4: Macro F1 = 0.3335
fold 5: Macro F1 = 0.3549
CV Macro F1 = 0.3364 (std 0.0142)

V4 실험 | 가중치 강도: 0.5
fold 1: Macro F1 = 0.3517
fold 2: Macro F1 = 0.3426
fold 3: Macro F1 = 0.3237
fold 4: Macro F1 = 0.3393
fold 5: Macro F1 = 0.3502
CV Macro F1 = 0.3415 (std 0.0100)

V4 실험 | 가중치 강도: 0.75
fold 1: Macro F1 = 0.3484
fold 2: Macro F1 = 0.3609
fold 3: Macro F1 = 0.3329
fold 4: Macro F1 = 0.3320
fold 5: Macro F1 = 0.3698
CV Macro F1 = 0.3488 (std 0.0150)

=== 전체 가중치 실험 비교 ===


,모델,가중치 강도,CV Macro F1,CV Std
0,V4_0.75,0.75,0.3488,0.0150
1,V3,1.00,0.3477,0.0158
2,V4_0.5,0.50,0.3415,0.0100
3,V4_0.25,0.25,0.3364,0.0142
4,V1,0.00,0.3280,0.0088



최고 성능 모델: V4_0.75
가중치 강도: 0.75
CV Macro F1: 0.3488


**V1·V3·V4 클래스별 비교:** 성능 개선이 일부 클래스에 집중되는지 확인하는 보조 분석입니다.

In [23]:
# V1/V3/V4 전체 암종별 비교
models_to_compare={'V1':lgbm_cv,'V4_0.25':v4_results[0.25],
                   'V4_0.50':v4_results[0.50],'V4_0.75':v4_results[0.75], 'V3':lgbm_cv_v3}
rows=[]
for model_name,result in models_to_compare.items():
    report=classification_report(y,result['oof'],labels=class_ids,target_names=class_names,output_dict=True,zero_division=0)
    counts=np.bincount(result['oof'],minlength=len(class_names))
    for idx,name in enumerate(class_names):
        rows.append({'모델':model_name,'암종':name,'실제 수':int(report[name]['support']),
                     '예측 수':int(counts[idx]),'Precision':report[name]['precision'],
                     'Recall':report[name]['recall'],'F1':report[name]['f1-score']})
detail_comparison=pd.DataFrame(rows)
display(detail_comparison.sort_values(['암종','모델']).round(4))


,모델,암종,실제 수,예측 수,Precision,Recall,F1
0,V1,ACC,72,47,0.9362,0.6111,0.7395
104,V3,ACC,72,55,0.8909,0.6806,0.7717
26,V4_0.25,ACC,72,49,0.8980,0.6111,0.7273
52,V4_0.50,ACC,72,51,0.8824,0.6250,0.7317
78,V4_0.75,ACC,72,55,0.8545,0.6528,0.7402
...,...,...,...,...,...,...,...
25,V1,UCEC,198,156,0.5769,0.4545,0.5085
129,V3,UCEC,198,200,0.5200,0.5253,0.5226
51,V4_0.25,UCEC,198,169,0.5740,0.4899,0.5286
77,V4_0.50,UCEC,198,172,0.5814,0.5051,0.5405


## 4. XGBoost 및 CatBoost 실험

**목적:** 동일한 전처리와 Fold에서 기본 모델과 학습 Fold별 balanced sample weight 모델을 비교합니다. 가중치는 검증 Fold의 정답을 사용하지 않습니다.

**XGBoost V1:** 평균 Macro F1 **0.3201**. LightGBM 기본 모델보다 높으나 V3 balanced보다 낮습니다.

In [24]:
# ============================================================
# XGBoost V1 - 기본 모델
# 기존 LightGBM과 동일한 전처리 / 동일한 5-Fold
# ============================================================

from xgboost import XGBClassifier

# 1. XGBoost 기본 파라미터
XGB_PARAMS = {
    "n_estimators": 200,
    "learning_rate": 0.05,
    "max_depth": 6,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "objective": "multi:softprob",
    "eval_metric": "mlogloss",
    "tree_method": "hist",
    "device": XGB_DEVICE,
    "random_state": SEED,
    "n_jobs": -1
}

# 2. 기존 팀 공통 전처리 사용
xgb_pipe = Pipeline([
    ("features", basic_transformer()),
    ("model", XGBClassifier(**XGB_PARAMS))
])

# 3. 기존과 동일한 5-Fold CV
xgb_cv = run_cv(
    xgb_pipe,
    X,
    y,
    folds=folds
)

# 4. 기존 모델과 성능 비교
print("\n=== 모델 성능 비교 ===")
print(f"LightGBM V1 : {lgbm_cv['mean']:.4f}")
print(f"LightGBM V3 : {lgbm_cv_v3['mean']:.4f}")
print(f"XGBoost V1 : {xgb_cv['mean']:.4f}")

print(
    f"\nXGBoost - LightGBM V3: "
    f"{xgb_cv['mean'] - lgbm_cv_v3['mean']:+.4f}"
)

c:\Users\user\Desktop\AI_Projects\ah07-cancer-subtype\.venv\Lib\site-packages\xgboost\core.py:569: UserWarning: [13:48:45] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


fold 1: Macro F1 = 0.3250
fold 2: Macro F1 = 0.3572
fold 3: Macro F1 = 0.3349
fold 4: Macro F1 = 0.3424
fold 5: Macro F1 = 0.3447
CV Macro F1 = 0.3408 (std 0.0107)

=== 모델 성능 비교 ===
LightGBM V1 : 0.3280
LightGBM V3 : 0.3477
XGBoost V1 : 0.3408

XGBoost - LightGBM V3: -0.0069


**CatBoost V1:** 평균 Macro F1 **0.2762**. 단독 성능은 낮지만 다른 모델과 오류 양상이 다를 수 있어 앙상블 후보로 보관합니다.

In [25]:
# ============================================================
# CatBoost V1 - 기본 모델
# - 기존 팀 공통 전처리 사용
# - 동일한 5-Fold
# - CPU 학습
# - 기존 코드 수정 없음
# ============================================================

from catboost import CatBoostClassifier

CAT_PARAMS = {
    "iterations": 300,
    "learning_rate": 0.05,
    "depth": 6,
    "loss_function": "MultiClass",
    "l2_leaf_reg": 3,
    "random_seed": SEED,
    "thread_count": -1,
    "task_type": "CPU",
    "verbose": False,
    "allow_writing_files": False
}

cat_pipe = Pipeline([
    ("features", basic_transformer()),
    ("model", CatBoostClassifier(**CAT_PARAMS))
])

# 동일한 5-Fold CV
cat_cv = run_cv(
    cat_pipe,
    X,
    y,
    folds=folds
)

# 결과 비교
comparison = pd.DataFrame([
    {
        "Model": "LightGBM V1",
        "Macro F1": lgbm_cv["mean"],
        "Std": lgbm_cv["std"]
    },
    {
        "Model": "LightGBM V3",
        "Macro F1": lgbm_cv_v3["mean"],
        "Std": lgbm_cv_v3["std"]
    },
    {
        "Model": "XGBoost V1",
        "Macro F1": xgb_cv["mean"],
        "Std": xgb_cv["std"]
    },
    {
        "Model": "CatBoost V1",
        "Macro F1": cat_cv["mean"],
        "Std": cat_cv["std"]
    }
])

display(
    comparison.sort_values(
        "Macro F1",
        ascending=False
    ).round(4)
)

fold 1: Macro F1 = 0.2898
fold 2: Macro F1 = 0.3090
fold 3: Macro F1 = 0.2720
fold 4: Macro F1 = 0.2909
fold 5: Macro F1 = 0.2664
CV Macro F1 = 0.2856 (std 0.0152)


,Model,Macro F1,Std
1,LightGBM V3,0.3477,0.0158
2,XGBoost V1,0.3408,0.0107
0,LightGBM V1,0.3280,0.0088
3,CatBoost V1,0.2856,0.0152


**공통 가중치 함수:** XGBoost와 CatBoost에서 동일한 Fold별 balanced sample weight를 사용합니다. 중복 정의를 하나로 합쳤습니다.

In [26]:
def balanced_fit_params(y_train):
    """훈련 Fold의 라벨만 사용해 가중치 계산."""
    return {"model__sample_weight": compute_sample_weight("balanced", y=y_train)}

**XGBoost V2:** 클래스 가중치 적용 후 5-Fold 평균 Macro F1 **0.3497**로 향상됐습니다. 개별 모델 중 최고 성능입니다.

In [27]:
# ============================================================
# XGBoost V2 - 클래스 불균형 가중치 적용
# 기존 XGBoost V1과 동일한 모델 설정 / Fold
# ============================================================


# 1. 기존 XGBoost 복제
xgb_v2 = clone(xgb_pipe)

# 2. 공통 가중치 함수를 사용합니다.
# 3. 동일한 5-Fold CV
xgb_cv_v2 = run_cv(
    xgb_v2,
    X,
    y,
    folds=folds,
    fit_params_fn=balanced_fit_params
)

# 4. 결과 비교
comparison_v2 = pd.DataFrame([
    {
        "모델": "LightGBM V3",
        "Macro F1": lgbm_cv_v3["mean"],
        "Std": lgbm_cv_v3["std"]
    },
    {
        "모델": "XGBoost V1",
        "Macro F1": xgb_cv["mean"],
        "Std": xgb_cv["std"]
    },
    {
        "모델": "XGBoost V2 Balanced",
        "Macro F1": xgb_cv_v2["mean"],
        "Std": xgb_cv_v2["std"]
    },
    {
        "모델": "CatBoost V1",
        "Macro F1": cat_cv["mean"],
        "Std": cat_cv["std"]
    }
])

display(
    comparison_v2.sort_values(
        "Macro F1",
        ascending=False
    ).round(4)
)

fold 1: Macro F1 = 0.3373
fold 2: Macro F1 = 0.3720
fold 3: Macro F1 = 0.3381
fold 4: Macro F1 = 0.3462
fold 5: Macro F1 = 0.3602
CV Macro F1 = 0.3508 (std 0.0134)


,모델,Macro F1,Std
2,XGBoost V2 Balanced,0.3508,0.0134
0,LightGBM V3,0.3477,0.0158
1,XGBoost V1,0.3408,0.0107
3,CatBoost V1,0.2856,0.0152


**CatBoost V2:** 가중치 적용 후 평균 Macro F1 **0.3189**로 개선됐습니다. 단독 최고 모델은 아니지만 앙상블에 기여하는지 평가합니다.

In [28]:
# ============================================================
# CatBoost V2 - Balanced Sample Weight
# 기존 CatBoost V1과 동일한 설정 및 5-Fold
# ============================================================


# 1. 기존 CatBoost 모델 복제
cat_v2 = clone(cat_pipe)

# 2. 공통 가중치 함수를 사용합니다.
# 3. 동일한 5-Fold CV
cat_cv_v2 = run_cv(
    cat_v2,
    X,
    y,
    folds=folds,
    fit_params_fn=balanced_fit_params
)

# 4. 모든 주요 모델 비교
model_results = {
    "LightGBM V1": lgbm_cv,
    "LightGBM V3 Balanced": lgbm_cv_v3,
    "XGBoost V1": xgb_cv,
    "XGBoost V2 Balanced": xgb_cv_v2,
    "CatBoost V1": cat_cv,
    "CatBoost V2 Balanced": cat_cv_v2
}

comparison_final = pd.DataFrame([
    {
        "모델": name,
        "Macro F1": result["mean"],
        "Std": result["std"]
    }
    for name, result in model_results.items()
])

display(
    comparison_final.sort_values(
        "Macro F1",
        ascending=False
    ).round(4)
)

fold 1: Macro F1 = 0.3320
fold 2: Macro F1 = 0.3386
fold 3: Macro F1 = 0.3084
fold 4: Macro F1 = 0.3285
fold 5: Macro F1 = 0.3205
CV Macro F1 = 0.3256 (std 0.0104)


,모델,Macro F1,Std
3,XGBoost V2 Balanced,0.3508,0.0134
1,LightGBM V3 Balanced,0.3477,0.0158
2,XGBoost V1,0.3408,0.0107
0,LightGBM V1,0.3280,0.0088
5,CatBoost V2 Balanced,0.3256,0.0104
4,CatBoost V1,0.2856,0.0152


## 5. Soft Voting 앙상블 및 오류 분석

**목적:** 서로 다른 모델의 OOF 예측 확률을 평균하여 최종 클래스를 결정합니다. 모든 확률 열은 동일한 클래스 순서여야 합니다. 조합 선택에 같은 OOF를 사용했으므로 최고 점수는 탐색적 결과이며 독립 테스트 검증을 대체하지 않습니다.

**앙상블 조합 비교:** XGB+CAT 전체 OOF Macro F1 **0.3595**, XGB+LGBM+CAT **0.3512**, XGB+LGBM **0.3425**. 동일 가중치 XGB+CAT를 최종 후보로 선택합니다.

In [29]:
# ============================================================
# Ensemble V1: Soft Voting (OOF 기반)
# 기존 학습 결과 재사용, 재학습 없음
# ============================================================

# 1. 각 모델의 OOF 확률
ensemble_models = {
    "XGBoost V2": xgb_cv_v2,
    "LightGBM V3": lgbm_cv_v3,
    "CatBoost V2": cat_cv_v2
}

# 2. OOF 확률 유효성 검사
for name, result in ensemble_models.items():
    proba = result["oof_proba"]

    assert proba is not None, f"{name}: OOF 확률 없음"
    assert proba.shape == (len(y), len(np.unique(y))), (
        f"{name}: OOF 확률 shape 불일치"
    )
    assert np.isfinite(proba).all(), (
        f"{name}: NaN 또는 무한대 존재"
    )

print("모든 모델의 OOF 확률 검증 완료")


# 3. Soft Voting 함수
def soft_voting(results, weights=None):

    probas = [
        result["oof_proba"]
        for result in results
    ]

    avg_proba = np.average(
        np.stack(probas),
        axis=0,
        weights=weights
    )

    predictions = avg_proba.argmax(axis=1)

    return predictions, avg_proba


# 4. 앙상블 조합 비교
ensemble_combinations = {
    "XGB + LGBM": [
        xgb_cv_v2,
        lgbm_cv_v3
    ],
    "XGB + CAT": [
        xgb_cv_v2,
        cat_cv_v2
    ],
    "XGB + LGBM + CAT": [
        xgb_cv_v2,
        lgbm_cv_v3,
        cat_cv_v2
    ]
}

ensemble_results = {}

for name, results in ensemble_combinations.items():

    pred, proba = soft_voting(results)

    score = f1_score(
        y,
        pred,
        average="macro",
        zero_division=0
    )

    ensemble_results[name] = {
        "macro_f1": score,
        "oof": pred,
        "oof_proba": proba
    }


# 5. 단일 모델과 앙상블 비교
comparison_rows = [
    {
        "모델": "XGBoost V2",
        "OOF Macro F1": f1_score(
            y, xgb_cv_v2["oof"],
            average="macro",
            zero_division=0
        )
    },
    {
        "모델": "LightGBM V3",
        "OOF Macro F1": f1_score(
            y, lgbm_cv_v3["oof"],
            average="macro",
            zero_division=0
        )
    },
    {
        "모델": "CatBoost V2",
        "OOF Macro F1": f1_score(
            y, cat_cv_v2["oof"],
            average="macro",
            zero_division=0
        )
    }
]

for name, result in ensemble_results.items():
    comparison_rows.append({
        "모델": name,
        "OOF Macro F1": result["macro_f1"]
    })

ensemble_comparison = pd.DataFrame(
    comparison_rows
).sort_values(
    "OOF Macro F1",
    ascending=False
)

display(ensemble_comparison.round(4))

모든 모델의 OOF 확률 검증 완료


,모델,OOF Macro F1
5,XGB + LGBM + CAT,0.3625
3,XGB + LGBM,0.3570
4,XGB + CAT,0.3561
0,XGBoost V2,0.3523
1,LightGBM V3,0.3495
2,CatBoost V2,0.3262


**앙상블 효과 분석:** XGBoost 단독 대비 정답 전환 **160건**, 오답 전환 **103건**, 순증가 **57건**. 클래스 26개 중 **16개 개선**, **10개 악화**로 개선이 균일하지는 않습니다.

In [30]:
# ============================================================
# 앙상블 분석: XGBoost V2 vs CatBoost V2 vs Soft Voting
# ============================================================

from sklearn.metrics import accuracy_score

y_true = np.asarray(y)

xgb_pred = np.asarray(xgb_cv_v2["oof"])
cat_pred = np.asarray(cat_cv_v2["oof"])
ensemble_pred = np.asarray(ensemble_results["XGB + CAT"]["oof"])

# 1. 각 모델의 정답 여부
xgb_correct = xgb_pred == y_true
cat_correct = cat_pred == y_true
ensemble_correct = ensemble_pred == y_true

print("=" * 60)
print("1. 모델별 정확도 및 Macro F1")
print("=" * 60)

performance = pd.DataFrame([
    {
        "모델": "XGBoost V2",
        "Accuracy": accuracy_score(y_true, xgb_pred),
        "Macro F1": f1_score(
            y_true, xgb_pred,
            average="macro", zero_division=0
        )
    },
    {
        "모델": "CatBoost V2",
        "Accuracy": accuracy_score(y_true, cat_pred),
        "Macro F1": f1_score(
            y_true, cat_pred,
            average="macro", zero_division=0
        )
    },
    {
        "모델": "XGB + CAT",
        "Accuracy": accuracy_score(y_true, ensemble_pred),
        "Macro F1": f1_score(
            y_true, ensemble_pred,
            average="macro", zero_division=0
        )
    }
])

display(performance.round(4))


print("\n" + "=" * 60)
print("2. XGBoost와 CatBoost의 정답 관계")
print("=" * 60)

correctness = pd.DataFrame([
    {
        "구분": "두 모델 모두 정답",
        "샘플 수": int((xgb_correct & cat_correct).sum())
    },
    {
        "구분": "XGBoost만 정답",
        "샘플 수": int((xgb_correct & ~cat_correct).sum())
    },
    {
        "구분": "CatBoost만 정답",
        "샘플 수": int((~xgb_correct & cat_correct).sum())
    },
    {
        "구분": "두 모델 모두 오답",
        "샘플 수": int((~xgb_correct & ~cat_correct).sum())
    }
])

display(correctness)


print("\n" + "=" * 60)
print("3. 앙상블로 개선되거나 악화된 샘플")
print("=" * 60)

improved = (~xgb_correct) & ensemble_correct
worsened = xgb_correct & (~ensemble_correct)

print("XGBoost 오답 → 앙상블 정답:", int(improved.sum()))
print("XGBoost 정답 → 앙상블 오답:", int(worsened.sum()))
print("정답 샘플 순증가:", int(improved.sum() - worsened.sum()))


print("\n" + "=" * 60)
print("4. 클래스별 F1 변화")
print("=" * 60)

xgb_report = classification_report(
    y_true, xgb_pred,
    output_dict=True, zero_division=0
)

ensemble_report = classification_report(
    y_true, ensemble_pred,
    output_dict=True, zero_division=0
)

class_rows = []

for cls in np.unique(y_true):
    key = str(cls)

    xgb_f1 = xgb_report[key]["f1-score"]
    ens_f1 = ensemble_report[key]["f1-score"]

    class_rows.append({
        "암종": le.classes_[int(cls)],
        "실제 수": int(xgb_report[key]["support"]),
        "XGB F1": xgb_f1,
        "앙상블 F1": ens_f1,
        "F1 변화": ens_f1 - xgb_f1
    })

class_comparison = pd.DataFrame(class_rows)

display(
    class_comparison.sort_values(
        "F1 변화",
        ascending=False
    ).round(4)
)

print(
    "\n개선된 클래스:",
    int((class_comparison["F1 변화"] > 0).sum())
)

print(
    "악화된 클래스:",
    int((class_comparison["F1 변화"] < 0).sum())
)

1. 모델별 정확도 및 Macro F1


,모델,Accuracy,Macro F1
0,XGBoost V2,0.3588,0.3523
1,CatBoost V2,0.3245,0.3262
2,XGB + CAT,0.3599,0.3561



2. XGBoost와 CatBoost의 정답 관계


,구분,샘플 수
0,두 모델 모두 정답,1717
1,XGBoost만 정답,508
2,CatBoost만 정답,295
3,두 모델 모두 오답,3681



3. 앙상블로 개선되거나 악화된 샘플
XGBoost 오답 → 앙상블 정답: 101
XGBoost 정답 → 앙상블 오답: 94
정답 샘플 순증가: 7

4. 클래스별 F1 변화


,암종,실제 수,XGB F1,앙상블 F1,F1 변화
5,DLBC,38,0.3714,0.4324,0.0610
13,LUAD,184,0.1994,0.2296,0.0303
14,LUSC,178,0.2671,0.2906,0.0235
10,LAML,158,0.4610,0.4830,0.0220
12,LIHC,158,0.2689,0.2877,0.0189
25,UCEC,198,0.5205,0.5342,0.0138
9,KIRC,334,0.4474,0.4602,0.0127
3,CESC,155,0.0932,0.1039,0.0107
22,TGCT,124,0.2581,0.2679,0.0098
20,SKCM,276,0.6449,0.6523,0.0074



개선된 클래스: 16
악화된 클래스: 10


**Fold별 안정성:** XGB+CAT 평균 Macro F1 **0.3586**, XGBoost V2 **0.3497**. 5개 Fold 모두 개선됐습니다. pandas의 기본 표본 표준편차(`ddof=1`)와 팀 공통 `run_cv()`의 모집단 표준편차(`ddof=0`)는 다릅니다.

In [31]:
# ============================================================
# Ensemble V2: Fold별 성능 안정성 분석
# - 모델 재학습 없음
# - 기존 OOF 결과 재사용
# - 동일한 5-Fold 검증
# ============================================================


models_for_fold = {
    "XGBoost V2": xgb_cv_v2["oof"],
    "LightGBM V3": lgbm_cv_v3["oof"],
    "CatBoost V2": cat_cv_v2["oof"],
    "XGB + CAT": ensemble_results["XGB + CAT"]["oof"],
    "XGB + LGBM + CAT": ensemble_results[
        "XGB + LGBM + CAT"
    ]["oof"]
}

fold_rows = []

for fold_idx, (train_idx, valid_idx) in enumerate(folds):

    for model_name, predictions in models_for_fold.items():

        score = f1_score(
            np.asarray(y)[valid_idx],
            np.asarray(predictions)[valid_idx],
            average="macro",
            zero_division=0
        )

        fold_rows.append({
            "Fold": fold_idx,
            "Model": model_name,
            "Macro F1": score
        })

fold_results = pd.DataFrame(fold_rows)

print("=== Fold별 성능 ===")

display(
    fold_results.pivot(
        index="Fold",
        columns="Model",
        values="Macro F1"
    ).round(4)
)

print("\n=== 모델별 평균 및 표준편차 ===")

fold_summary = (
    fold_results
    .groupby("Model")["Macro F1"]
    .agg(["mean", "std", "min", "max"])
    .sort_values("mean", ascending=False)
)

display(fold_summary.round(4))

print("\n=== XGBoost 대비 앙상블 Fold별 개선 ===")

fold_pivot = fold_results.pivot(
    index="Fold",
    columns="Model",
    values="Macro F1"
)

fold_pivot["Ensemble Gain"] = (
    fold_pivot["XGB + CAT"]
    - fold_pivot["XGBoost V2"]
)

display(
    fold_pivot[
        ["XGBoost V2", "XGB + CAT", "Ensemble Gain"]
    ].round(4)
)

=== Fold별 성능 ===


Model,CatBoost V2,LightGBM V3,XGB + CAT,XGB + LGBM + CAT,XGBoost V2
Fold,,,,,
0,0.3320,0.3430,0.3378,0.3425,0.3373
1,0.3386,0.3560,0.3793,0.3811,0.3720
2,0.3084,0.3328,0.3310,0.3472,0.3381
3,0.3285,0.3325,0.3499,0.3467,0.3462
4,0.3205,0.3742,0.3767,0.3840,0.3602



=== 모델별 평균 및 표준편차 ===


,mean,std,min,max
Model,,,,
XGB + LGBM + CAT,0.3603,0.0204,0.3425,0.3840
XGB + CAT,0.3550,0.0221,0.3310,0.3793
XGBoost V2,0.3508,0.0150,0.3373,0.3720
LightGBM V3,0.3477,0.0177,0.3325,0.3742
CatBoost V2,0.3256,0.0116,0.3084,0.3386



=== XGBoost 대비 앙상블 Fold별 개선 ===


Model,XGBoost V2,XGB + CAT,Ensemble Gain
Fold,,,
0,0.3373,0.3378,0.0005
1,0.3720,0.3793,0.0074
2,0.3381,0.3310,-0.0071
3,0.3462,0.3499,0.0037
4,0.3602,0.3767,0.0166


## 6. 전체 학습 및 테스트 예측

**목적:** 선정된 모델 두 개를 전체 학습 데이터로 다시 학습합니다. 테스트 정답은 사용하지 않습니다. 원본 학습 결과의 테스트 확률 형태는 **(2546, 26)**이며 26개 암종 예측을 생성했습니다. 실제 제출 라벨 변환은 팀 공통 인코더 규칙에 따라 별도 수행해야 합니다.

**최종 추론:** XGBoost V2와 CatBoost V2를 전체 학습 데이터로 학습하고 확률을 1:1 평균합니다. 모델 파라미터와 전처리는 CV 실험과 동일합니다.

In [32]:
# ============================================================
# 최종 후보: XGBoost V2 + CatBoost V2
# 전체 Train 학습 및 Test 예측 확률 생성
# ============================================================


# 1. 최종 모델 복제
final_xgb = clone(xgb_pipe)
final_cat = clone(cat_pipe)

# 2. 전체 학습 데이터 기준 클래스 가중치
full_weights = compute_sample_weight(
    class_weight="balanced",
    y=y
)

# 3. 전체 데이터 학습
print("XGBoost 전체 학습 시작")

final_xgb.fit(
    X,
    y,
    model__sample_weight=full_weights
)

print("XGBoost 전체 학습 완료")

print("\nCatBoost 전체 학습 시작")

final_cat.fit(
    X,
    y,
    model__sample_weight=full_weights
)

print("CatBoost 전체 학습 완료")

# 4. Test 특성 준비
# test는 기존에 로딩한 원본 테스트 DataFrame
X_test = test[X.columns].copy()

assert X_test.columns.equals(X.columns)
assert len(X_test) == len(test)

# 5. 테스트 확률 예측
xgb_test_proba = final_xgb.predict_proba(X_test)
cat_test_proba = final_cat.predict_proba(X_test)

assert xgb_test_proba.shape == cat_test_proba.shape
assert xgb_test_proba.shape[1] == len(np.unique(y))

# 6. 동일 가중치 Soft Voting
final_test_proba = (
    xgb_test_proba + cat_test_proba
) / 2

final_test_pred = final_test_proba.argmax(axis=1)

# 7. 결과 검증
assert np.isfinite(final_test_proba).all()
assert np.allclose(final_test_proba.sum(axis=1), 1, atol=1e-5)

print("\n=== 최종 예측 검증 ===")
print("Test 샘플 수:", len(X_test))
print("확률 행렬:", final_test_proba.shape)
print("최종 예측:", final_test_pred.shape)

print("\n예측 클래스 분포:")
print(
    dict(
        zip(
            *np.unique(
                final_test_pred,
                return_counts=True
            )
        )
    )
)

XGBoost 전체 학습 시작
XGBoost 전체 학습 완료

CatBoost 전체 학습 시작
CatBoost 전체 학습 완료

=== 최종 예측 검증 ===
Test 샘플 수: 2546
확률 행렬: (2546, 26)
최종 예측: (2546,)

예측 클래스 분포:
{np.int64(0): np.int64(22), np.int64(1): np.int64(62), np.int64(2): np.int64(152), np.int64(3): np.int64(58), np.int64(4): np.int64(578), np.int64(5): np.int64(18), np.int64(6): np.int64(66), np.int64(7): np.int64(58), np.int64(8): np.int64(49), np.int64(9): np.int64(74), np.int64(10): np.int64(51), np.int64(11): np.int64(95), np.int64(12): np.int64(42), np.int64(13): np.int64(41), np.int64(14): np.int64(55), np.int64(15): np.int64(173), np.int64(16): np.int64(38), np.int64(17): np.int64(93), np.int64(18): np.int64(47), np.int64(19): np.int64(34), np.int64(20): np.int64(109), np.int64(21): np.int64(108), np.int64(22): np.int64(85), np.int64(23): np.int64(73), np.int64(24): np.int64(251), np.int64(25): np.int64(114)}


## 7. 결론 및 재현성 점검

- **최종 후보:** XGBoost V2 + CatBoost V2 (동일 가중치 Soft Voting)
- **검증:** 5-Fold 평균 Macro F1 **0.3586**, 전체 OOF Macro F1 **0.3595** (업로드 원본 실행 기록)
- **기준 모델:** XGBoost V2 평균 **0.3497**, 전체 OOF **0.3504**
- **한계:** 동일 OOF를 여러 실험과 모델 선택에 반복 활용했으므로 독립 성능 추정치가 아님. 테스트의 클래스 분포도 정답 분포를 뜻하지 않음
- **재현:** 팀 공통 `src` 모듈, 원본 `train.csv`/`test.csv`, 동일 패키지 버전 및 `SEED`가 필요합니다. 본 파일은 업로드된 실행 기록을 보존했으나 수정 셀과 전체 노트북을 실제 환경에서 처음부터 재실행해 검증해야 합니다.
- **팀 공유:** 공통 코드의 변경 없이 담당 모델링 셀을 정리했습니다. 최종 제출 전 클래스 라벨 복원 및 제출 스키마를 팀 담당자와 확인하세요.

## 8. 11월 7일 실험 기록 — 팀 공통 함수 사용

**목적:** `run_cv()`로 이미 얻은 결과를 `src.evaluation.log_result()`로 `experiments/log_A_model.csv`에 저장합니다. 원본 실험·모델 파라미터·CV 결과는 변경하지 않습니다.

- 기록 날짜는 **코드를 실행한 실제 시각**입니다. 실험 식별자에는 `1107`을 포함합니다.
- `log_result()`는 CSV에 **추가(append)** 하므로 같은 실험 ID가 있으면 재기록하지 않습니다.
- XGBoost/CatBoost V2는 **기본 모델 파라미터 + Fold별 `balanced_fit_params()`**를 사용했습니다. 모델 객체의 파라미터만 기록하면 가중치 설정이 누락되므로 `notes`에도 명시합니다.
- LightGBM V4는 각 가중치 강도마다 별도 실험으로 기록합니다.
- **기록은 5-Fold 전체 실행 결과만** 허용합니다. 이 셀은 재학습하지 않습니다.
- 기존 LightGBM CV는 `deterministic=True, force_row_wise=True`가 없는 설정으로 실행되었습니다. 재현성 옵션을 적용한 새 실험은 별도 ID로 수행해야 하며, 기존 점수를 새 설정의 점수로 기록하면 안 됩니다.


In [33]:
# 11/07 실험 결과를 팀 공통 CSV에 기록 (재실행 시 중복 방지)
# 사전 조건: 위의 5-Fold CV 셀들을 정상 실행했을 것
from src.config import EXPERIMENT_DIR

OWNER = "A_model"
LOG_PATH = EXPERIMENT_DIR / f"log_{OWNER}.csv"

# 원본 CV 결과의 실제 모델 설정을 그대로 기록합니다.
experiment_specs = [
    ("1107_lgbm_v1", "LightGBM", lgbm_pipe, lgbm_cv, "기본 모델"),
    ("1107_lgbm_v2", "LightGBM", lgbm_v2, lgbm_cv_v2, "n_estimators=400, learning_rate=0.03, num_leaves=31, max_depth=8, min_child_samples=30"),
    ("1107_lgbm_v3", "LightGBM", lgbm_v3, lgbm_cv_v3, "class_weight=balanced"),
    ("1107_xgb_v1", "XGBoost", xgb_pipe, xgb_cv, "기본 모델"),
    ("1107_xgb_v2", "XGBoost", xgb_v2, xgb_cv_v2, "Fold 학습 라벨 기준 balanced sample_weight; fit_params_fn=balanced_fit_params"),
    ("1107_cat_v1", "CatBoost", cat_pipe, cat_cv, "기본 모델"),
    ("1107_cat_v2", "CatBoost", cat_v2, cat_cv_v2, "Fold 학습 라벨 기준 balanced sample_weight; fit_params_fn=balanced_fit_params"),
]

for strength in (0.25, 0.50, 0.75):
    experiment_specs.append((
        f"1107_lgbm_v4_{int(strength * 100):03d}",
        "LightGBM", lgbm_pipe, v4_results[strength],
        f"Fold 학습 라벨 기준 부분 가중치 strength={strength}; class_weight=None; device_type=cpu"
    ))

existing_ids = set()
if LOG_PATH.exists() and LOG_PATH.stat().st_size > 0:
    existing_df = pd.read_csv(LOG_PATH, encoding="utf-8-sig")
    if "experiment" not in existing_df.columns:
        raise ValueError(f"기존 로그에 experiment 열이 없습니다: {LOG_PATH}")
    existing_ids = set(existing_df["experiment"].dropna().astype(str))

for exp_id, model_name, pipe, cv_result, note in experiment_specs:
    if exp_id in existing_ids:
        print(f"기록 유지(중복 생략): {exp_id}")
        continue
    if len(cv_result["scores"]) != len(folds) or len(folds) != 5:
        raise ValueError(f"{exp_id}: 5-Fold 전체 결과만 기록할 수 있습니다.")
    params = pipe.named_steps["model"].get_params().copy()
    if exp_id.startswith("1107_lgbm_v4"):
        params.update(class_weight=None, device_type="cpu", weight_strength=float(note.split("strength=")[1].split(";")[0]))
    if exp_id.endswith("xgb_v2") or exp_id.endswith("cat_v2"):
        params["fit_sample_weight"] = "balanced_per_training_fold"
    log_result(
        owner=OWNER,
        notebook="notebooks/A_model/03_models.ipynb",
        experiment=exp_id,
        features="WT_binary (basic_transformer)",
        model=model_name,
        params=params,
        cv=cv_result,
        notes=f"11/07 실험 | seed={SEED} | {note}"
    )
    existing_ids.add(exp_id)

print("\n현재 기록된 실험 결과:")
display(pd.read_csv(LOG_PATH, encoding="utf-8-sig")[["experiment", "model", "cv_mean", "cv_std", "notes"]])

기록 유지(중복 생략): 1107_lgbm_v1
기록 유지(중복 생략): 1107_lgbm_v2
기록 유지(중복 생략): 1107_lgbm_v3
기록 유지(중복 생략): 1107_xgb_v1
기록 유지(중복 생략): 1107_xgb_v2
기록 유지(중복 생략): 1107_cat_v1
기록 유지(중복 생략): 1107_cat_v2
기록 유지(중복 생략): 1107_lgbm_v4_025
기록 유지(중복 생략): 1107_lgbm_v4_050
기록 유지(중복 생략): 1107_lgbm_v4_075

현재 기록된 실험 결과:


,experiment,model,cv_mean,cv_std,notes
0,1107_lgbm_v1,LightGBM,0.3030,0.0087,11/07 실험 | seed=42 | 기본 모델
1,1107_lgbm_v2,LightGBM,0.3004,0.0109,"11/07 실험 | seed=42 | n_estimators=400, learnin..."
2,1107_lgbm_v3,LightGBM,0.3286,0.0101,11/07 실험 | seed=42 | class_weight=balanced
3,1107_xgb_v1,XGBoost,0.3201,0.0127,11/07 실험 | seed=42 | 기본 모델
4,1107_xgb_v2,XGBoost,0.3497,0.0033,11/07 실험 | seed=42 | Fold 학습 라벨 기준 balanced sa...
5,1107_cat_v1,CatBoost,0.2762,0.0094,11/07 실험 | seed=42 | 기본 모델
6,1107_cat_v2,CatBoost,0.3189,0.0078,11/07 실험 | seed=42 | Fold 학습 라벨 기준 balanced sa...
7,1107_lgbm_v4_025,LightGBM,0.3118,0.0098,11/07 실험 | seed=42 | Fold 학습 라벨 기준 부분 가중치 stre...
8,1107_lgbm_v4_050,LightGBM,0.3224,0.0123,11/07 실험 | seed=42 | Fold 학습 라벨 기준 부분 가중치 stre...
9,1107_lgbm_v4_075,LightGBM,0.3251,0.0128,11/07 실험 | seed=42 | Fold 학습 라벨 기준 부분 가중치 stre...


## 9. 앙상블 후보 확률 저장

**목적:** `05_ensemble.ipynb`에서 같은 행·같은 클래스 순서의 OOF 확률을 사용할 수 있도록 **XGBoost V2와 CatBoost V2**의 OOF/Test 확률을 팀 표준 형식으로 저장합니다.

- OOF 확률은 각 검증 Fold에서만 예측한 값이며, 테스트 확률은 **전체 Train으로 학습한 동일 설정의 모델**에서 얻은 값입니다.
- `le.classes_`는 원래 SUBCLASS 문자열의 순서이며, 모델 확률의 열 순서(인코딩된 0~25)와 대응합니다.
- 기존 파일명이 같으면 덮어쓰므로 **최종 후보의 확률을 저장할 때만** 실행하세요.
- `experiments/preds/`는 팀 README상 공유 대상입니다. 단, 파일 크기·대회 규정·`.gitignore`를 커밋 전에 확인하세요.
- 테스트 예측 분포를 보고 모델 가중치나 파라미터를 변경하지 않습니다.


In [34]:
# 앙상블 후보 모델의 OOF/Test 확률 저장 (재학습 없음)
class_ids = np.arange(len(le.classes_))
assert np.array_equal(np.unique(y), class_ids), "타깃 인코딩이 0..K-1인지 확인하세요"
assert np.array_equal(final_xgb.named_steps["model"].classes_, class_ids)
assert np.array_equal(final_cat.named_steps["model"].classes_, class_ids)

for name, cv_result, test_proba in [
    ("xgb_v2_balanced_1107", xgb_cv_v2, xgb_test_proba),
    ("cat_v2_balanced_1107", cat_cv_v2, cat_test_proba),
]:
    oof = np.asarray(cv_result["oof_proba"])
    test_probs = np.asarray(test_proba)
    assert oof.shape == (len(y), len(class_ids)), f"{name}: OOF shape 오류"
    assert test_probs.shape == (len(test), len(class_ids)), f"{name}: Test shape 오류"
    assert np.isfinite(oof).all() and np.isfinite(test_probs).all()
    assert np.allclose(oof.sum(axis=1), 1, atol=1e-4)
    assert np.allclose(test_probs.sum(axis=1), 1, atol=1e-4)
    save_predictions(
        owner="A_model",
        name=name,
        oof_proba=oof,
        test_proba=test_probs,
        classes=le.classes_,
        cv=cv_result,
        notes="2026-11-07 | WT_binary | balanced sample_weight | 5-Fold | CPU"
    )

# 저장된 두 후보를 읽어 파일/클래스/행 수가 올바른지 확인
saved = load_predictions(classes=le.classes_)
for name in ("xgb_v2_balanced_1107", "cat_v2_balanced_1107"):
    key = f"A_model__{name}"
    assert key in saved
    assert saved[key]["oof"].shape == (len(y), len(class_ids))
    assert saved[key]["test"].shape == (len(test), len(class_ids))
print("앙상블 후보 2개 저장 및 재로딩 확인 완료")

예측 저장: A_model__xgb_v2_balanced_1107 (CV 0.3508) → experiments/preds/
예측 저장: A_model__cat_v2_balanced_1107 (CV 0.3256) → experiments/preds/
불러온 예측 3개: ['A_model__cat_v2_balanced_1107', 'A_model__xgb_v2_balanced_1107', 'J_leader__baseline_xgb']
앙상블 후보 2개 저장 및 재로딩 확인 완료


## 10. 제출 전 체크리스트

1. **Kernel 재시작 → Run All**로 5-Fold 결과와 로그 저장까지 오류 없이 재현되는지 확인합니다.
2. `experiments/log_A_model.csv`의 모델별 CV 값이 실제 실행 출력과 일치하는지 확인합니다. `log_result()`의 `date`는 기록 실행 시각입니다.
3. `experiments/preds/`의 OOF/Test 확률 및 메타데이터 파일이 정상 생성됐는지 확인합니다.
4. 이 노트북에서 `src/` 공통 코드를 수정하지 않았습니다. `src` 변경이 필요하면 담당자와 공유합니다.
5. **원본 train/test CSV, 모델 파일, 제출 CSV, 가상환경은 커밋하지 않습니다.**
6. `feat/model_A_1107`에서 본인 노트북과 `experiments/log_A_model.csv`, 담당자 예측 파일만 스테이징한 뒤 `dev` 대상으로 PR을 올립니다. 팀원 1명 승인 후 Merge합니다.

> 기존 결과 수치는 과거 실행 기록이며, 이 파일을 제작하면서 실제 데이터로 전체 학습을 다시 수행한 것은 아닙니다. 재실행 후 결과를 확인하세요.


## 11/09 — 쌍둥이 Group K-Fold + 변이 유형 피처 비교

`src/validation.py`의 Group K-Fold를 사용합니다. `src/features.py`는 홍시윤님 담당 파일이므로 수정하지 않았습니다. 노트북 내부에서만 결측값을 임시로 WT로 처리하며, 공통 피처 수정이 병합되면 결과 일치 여부를 재검증하세요. 기존 실험 Markdown의 점수는 과거 기록이며 새 Group Fold 결과와 직접 비교할 수 없습니다.


In [35]:
from src.features import mutation_count, mutation_type_counts
from src.evaluation import save_predictions
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

class MutationFeatureTransformer(BaseEstimator, TransformerMixin):
    """원본 유전자 DataFrame -> 0/1 유전자 + 환자별 변이 개수 6개"""
    def fit(self, X, y=None):
        self.feature_names_in_ = list(X.columns)
        return self
    def transform(self, X):
        # 임시 호환 처리: src/features.py 담당자의 dropna 반영 전까지 사용.
        # 원본 X는 변경하지 않으며 쌍둥이 판정은 반드시 원본 X로 수행.
        X_features = X.fillna('WT')
        binary = binarize_mutations(X_features)
        counts = mutation_type_counts(X_features)
        total = mutation_count(X_features)
        return pd.concat([binary, total, counts], axis=1).astype('float32')

# 이 셀은 src/validation.py 교체 후 실행해야 합니다.
group_folds = get_folds(y, X=X, classes=le.classes_)
print('Group CV Fold 수:', len(group_folds))
print('새 피처 수:', MutationFeatureTransformer().fit_transform(X.iloc[:2]).shape[1])

Group CV Fold 수: 5
새 피처 수: 4390


In [36]:
# 3개 모델을 완전히 동일한 Group Fold로 평가합니다.
# 새 피처를 반영한 모델이므로 기존 기본 모델 점수와 구분해서 기록합니다.
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

MODELS_1109 = {
    'xgb_group_counts_1109': XGBClassifier(
        n_estimators=300, max_depth=5, learning_rate=0.05,
        subsample=0.9, colsample_bytree=0.9,
        objective='multi:softprob', eval_metric='mlogloss',
        random_state=SEED, n_jobs=-1, tree_method='hist', device=XGB_DEVICE),
    'lgbm_group_counts_1109': LGBMClassifier(
        n_estimators=300, learning_rate=0.05, num_leaves=15,
        verbosity=-1, random_state=SEED, n_jobs=-1),
    'lr_group_counts_1109': LogisticRegression(
        max_iter=2500, C=1.0, class_weight='balanced', random_state=SEED),
}

RESULTS_1109 = {}
# 비교는 동일한 group_folds로 수행. 기본 binary-only vs 파생피처 포함.
BASELINE_RESULTS_1109 = {}
for name, estimator in MODELS_1109.items():
    steps = [('features', MutationFeatureTransformer())]
    if name.startswith('lr_'):
        steps.append(('scale', StandardScaler()))
    steps.append(('model', estimator))
    pipe = Pipeline(steps)
    print('\n===', name, '===')
    cv = run_cv(pipe, X, y, folds=group_folds)
    RESULTS_1109[name] = {'model': pipe, 'cv': cv}
    baseline_steps = [('binary', basic_transformer())]
    if name.startswith('lr_'):
        baseline_steps.append(('scale', StandardScaler()))
    baseline_steps.append(('model', clone(estimator)))
    baseline_pipe = Pipeline(baseline_steps)
    print('=== 동일 Group Fold 기본 피처 비교:', name, '===')
    baseline_cv = run_cv(baseline_pipe, X, y, folds=group_folds)
    BASELINE_RESULTS_1109[name] = {'model': baseline_pipe, 'cv': baseline_cv}

display(pd.DataFrame([{'모델': k, '기본 Macro F1':BASELINE_RESULTS_1109[k]['cv']['mean'],
    '파생피처 Macro F1':v['cv']['mean'],
    '차이':v['cv']['mean']-BASELINE_RESULTS_1109[k]['cv']['mean']}
    for k,v in RESULTS_1109.items()]).sort_values('파생피처 Macro F1',ascending=False).round(4))


=== xgb_group_counts_1109 ===
fold 1: Macro F1 = 0.3993
fold 2: Macro F1 = 0.4368
fold 3: Macro F1 = 0.3771
fold 4: Macro F1 = 0.4289
fold 5: Macro F1 = 0.4497
CV Macro F1 = 0.4183 (std 0.0265)
=== 동일 Group Fold 기본 피처 비교: xgb_group_counts_1109 ===
fold 1: Macro F1 = 0.3387
fold 2: Macro F1 = 0.3585
fold 3: Macro F1 = 0.3411
fold 4: Macro F1 = 0.3378
fold 5: Macro F1 = 0.3534
CV Macro F1 = 0.3459 (std 0.0084)

=== lgbm_group_counts_1109 ===
fold 1: Macro F1 = 0.3976
fold 2: Macro F1 = 0.4171
fold 3: Macro F1 = 0.3476
fold 4: Macro F1 = 0.4022
fold 5: Macro F1 = 0.4085
CV Macro F1 = 0.3946 (std 0.0244)
=== 동일 Group Fold 기본 피처 비교: lgbm_group_counts_1109 ===
fold 1: Macro F1 = 0.3467
fold 2: Macro F1 = 0.3442
fold 3: Macro F1 = 0.2965
fold 4: Macro F1 = 0.3173
fold 5: Macro F1 = 0.3266
CV Macro F1 = 0.3263 (std 0.0185)

=== lr_group_counts_1109 ===
fold 1: Macro F1 = 0.2541
fold 2: Macro F1 = 0.2449
fold 3: Macro F1 = 0.2314
fold 4: Macro F1 = 0.2554
fold 5: Macro F1 = 0.2518
CV Macro F1 

,모델,기본 Macro F1,파생피처 Macro F1,차이
0,xgb_group_counts_1109,0.3459,0.4183,0.0725
1,lgbm_group_counts_1109,0.3263,0.3946,0.0683
2,lr_group_counts_1109,0.2518,0.2475,-0.0042


### 최종 제출 후보 CPU 재현성 검증 (선택 실행)

최종 모델 확정 후 `RUN_FINAL_CPU_CHECK=True`로 바꾸고 실행하세요. GPU 실험 결과와 동일 Group Fold에서 CPU Macro F1 및 Fold별 차이를 비교합니다.

In [37]:
# 최종 제출 후보가 결정되었을 때만 실행하세요. GPU와 CPU는 동일한 group_folds 사용.
# 기본값 False: 일반 Run All에서 추가 5-Fold 학습이 발생하지 않도록 방지.
RUN_FINAL_CPU_CHECK = False
FINAL_CANDIDATE_NAME = 'xgb_group_counts_1109'  # 최종 모델 선정 후 이름 변경

if RUN_FINAL_CPU_CHECK:
    if FINAL_CANDIDATE_NAME not in RESULTS_1109:
        raise KeyError(f"후보가 RESULTS_1109에 없습니다: {FINAL_CANDIDATE_NAME}")
    gpu_or_auto = RESULTS_1109[FINAL_CANDIDATE_NAME]
    candidate_cpu = clone(gpu_or_auto['model'])
    if 'model__device' not in candidate_cpu.get_params():
        raise ValueError('선택 모델에 XGBoost device 파라미터가 없습니다. CPU 검증 코드를 해당 모델에 맞게 수정하세요.')
    candidate_cpu.set_params(model__device='cpu')
    cpu_cv = run_cv(candidate_cpu, X, y, folds=group_folds)
    ref_cv = gpu_or_auto['cv']
    print('후보:', FINAL_CANDIDATE_NAME)
    print('기존 장치:', XGB_DEVICE, '| CV Macro F1:', round(ref_cv['mean'], 5))
    print('CPU 재실행 CV Macro F1:', round(cpu_cv['mean'], 5))
    print('차이 (CPU - 기존):', round(cpu_cv['mean']-ref_cv['mean'], 5))
    print('각 Fold 기존:', np.round(ref_cv['scores'], 5))
    print('각 Fold CPU :', np.round(cpu_cv['scores'], 5))
    # CPU 최종 제출 후보 전체 데이터 재학습; 실제 제출 모델 확정 후 사용
    final_cpu_model = clone(candidate_cpu).fit(X, y)
    print('CPU 전체 데이터 학습 성공:', type(final_cpu_model.named_steps['model']).__name__)
else:
    print('CPU 재현성 검증 대기: 최종 후보 확정 후 RUN_FINAL_CPU_CHECK=True로 실행')


CPU 재현성 검증 대기: 최종 후보 확정 후 RUN_FINAL_CPU_CHECK=True로 실행


### 11/09 전체 암종 이름·오분류·COAD 쏠림 (Train OOF만)

Test 예측의 암종 분포는 **확인하지 않습니다**. 26개 암종을 모두 이름으로 표시하고, 실제/예측 분포와 COAD로 잘못 분류된 암종을 비교합니다.


In [38]:
from sklearn.metrics import classification_report, confusion_matrix
class_names=np.asarray(le.classes_)
class_ids=np.arange(len(class_names))
for name,result in RESULTS_1109.items():
    pred=np.asarray(result['cv']['oof'],dtype=int)
    print('\n===',name,'전체 암종별 OOF 성능 ===')
    display(pd.DataFrame(classification_report(y,pred,labels=class_ids,target_names=class_names,
                                               output_dict=True,zero_division=0)).T.round(4))
    actual=np.bincount(y,minlength=len(class_names))
    estimated=np.bincount(pred,minlength=len(class_names))
    dist=pd.DataFrame({'암종':class_names,'실제 건수':actual,'OOF 예측 건수':estimated})
    dist['실제 비율']=dist['실제 건수']/len(y)
    dist['예측 비율']=dist['OOF 예측 건수']/len(y)
    dist['예측-실제 비율']=dist['예측 비율']-dist['실제 비율']
    display(dist.sort_values('예측-실제 비율',ascending=False).round(4))
    cm=confusion_matrix(y,pred,labels=class_ids)
    errors=[{'실제 암종':class_names[i],'예측 암종':class_names[j],'건수':int(cm[i,j])}
            for i in class_ids for j in class_ids if i!=j and cm[i,j]>0]
    display(pd.DataFrame(errors).sort_values('건수',ascending=False).head(20))
    if 'COAD' in class_names:
        coad_id=int(np.flatnonzero(class_names=='COAD')[0])
        print('COAD 실제 건수:',actual[coad_id],'OOF 예측 건수:',estimated[coad_id])
        coad_wrong=[{'실제 암종':class_names[i],'COAD로 오분류':int(cm[i,coad_id])}
                    for i in class_ids if i!=coad_id and cm[i,coad_id]>0]
        display(pd.DataFrame(coad_wrong).sort_values('COAD로 오분류',ascending=False))



=== xgb_group_counts_1109 전체 암종별 OOF 성능 ===


,precision,recall,f1-score,support
ACC,0.8929,0.6944,0.7812,72.0000
BLCA,0.5079,0.3077,0.3832,104.0000
BRCA,0.4436,0.5407,0.4874,786.0000
CESC,0.3377,0.1677,0.2241,155.0000
COAD,0.6820,0.6637,0.6727,223.0000
DLBC,0.9333,0.3684,0.5283,38.0000
GBMLGG,0.4759,0.4707,0.4733,461.0000
HNSC,0.3401,0.2242,0.2703,223.0000
KIPAN,0.3091,0.5029,0.3829,515.0000
KIRC,0.4121,0.2246,0.2907,334.0000


,암종,실제 건수,OOF 예측 건수,실제 비율,예측 비율,예측-실제 비율
8,KIPAN,515,838,0.0831,0.1351,0.0521
2,BRCA,786,958,0.1268,0.1545,0.0277
15,OV,253,377,0.0408,0.0608,0.0200
21,STES,379,451,0.0611,0.0727,0.0116
18,PRAD,266,334,0.0429,0.0539,0.0110
24,THYM,98,139,0.0158,0.0224,0.0066
17,PCPG,147,165,0.0237,0.0266,0.0029
14,LUSC,178,195,0.0287,0.0314,0.0027
20,SKCM,276,288,0.0445,0.0464,0.0019
10,LAML,158,154,0.0255,0.0248,-0.0006


,실제 암종,예측 암종,건수
157,KIRC,KIPAN,168
180,LGG,GBMLGG,102
101,GBMLGG,LGG,96
137,KIPAN,KIRC,81
32,BRCA,OV,77
132,KIPAN,BRCA,70
285,PRAD,BRCA,68
35,BRCA,PRAD,53
25,BRCA,KIPAN,50
304,SARC,BRCA,48


COAD 실제 건수: 223 OOF 예측 건수: 217


,실제 암종,COAD로 오분류
16,STES,12
11,OV,7
5,KIPAN,6
1,BRCA,5
2,CESC,5
4,HNSC,5
10,LUSC,4
8,LIHC,4
6,KIRC,4
12,PAAD,3


... (8 outputs left out)

,암종,실제 건수,OOF 예측 건수,실제 비율,예측 비율,예측-실제 비율
24,THYM,98,730,0.0158,0.1177,0.1019
10,LAML,158,386,0.0255,0.0622,0.0368
23,THCA,324,482,0.0522,0.0777,0.0255
17,PCPG,147,296,0.0237,0.0477,0.0240
18,PRAD,266,377,0.0429,0.0608,0.0179
2,BRCA,786,874,0.1268,0.1409,0.0142
19,SARC,198,227,0.0319,0.0366,0.0047
11,LGG,229,238,0.0369,0.0384,0.0015
15,OV,253,245,0.0408,0.0395,-0.0013
5,DLBC,38,23,0.0061,0.0037,-0.0024


,실제 암종,예측 암종,건수
54,BRCA,THYM,109
122,GBMLGG,LGG,105
166,KIPAN,KIRC,92
187,KIRC,KIPAN,76
454,THCA,THYM,75
360,PRAD,THYM,72
344,PCPG,THYM,69
215,LGG,GBMLGG,68
405,STES,BRCA,67
180,KIPAN,THYM,55


COAD 실제 건수: 223 OOF 예측 건수: 149


,실제 암종,COAD로 오분류
15,STES,19
4,HNSC,11
9,LUSC,9
16,UCEC,6
14,SKCM,6
8,LUAD,4
1,BRCA,3
10,OV,3
0,BLCA,2
2,CESC,2


### 실험별 OOF/Test 저장 (기본·파생피처 6개 실험)

위 모델 검증이 모두 성공한 후 실행합니다. Test는 **추론 및 예측 파일 저장에만** 사용합니다. 예측 분포나 정답을 조사하지 않습니다. `src/features.py`는 담당자 병합 이후 다시 확인하세요.


In [39]:
from src.config import EXPERIMENT_DIR
log_path=EXPERIMENT_DIR/'log_A_model.csv'
# 서로 같은 모델 이름의 baseline/derived가 덮어써지지 않도록 별도로 순회
experiments_to_save = (
    [(name+'_binary',result,'binary_only') for name,result in BASELINE_RESULTS_1109.items()] +
    [(name+'_derived',result,'binary+mutation_count+mutation_type_counts') for name,result in RESULTS_1109.items()]
)
for experiment_name,result,feature_description in experiments_to_save:
    full_model=clone(result['model']).fit(X,y)
    test_proba=full_model.predict_proba(test[X.columns])
    cv=result['cv']
    assert cv['oof_proba'] is not None
    assert test_proba.shape==(len(test),len(le.classes_))
    save_predictions('A_model',experiment_name,cv['oof_proba'],test_proba,
                     le.classes_,cv,notes='group CV; '+feature_description)
    already=False
    if log_path.exists():
        prior=pd.read_csv(log_path)
        already='experiment' in prior.columns and experiment_name in prior['experiment'].astype(str).values
    if not already:
        log_result(owner='A_model',notebook='03_models.ipynb',experiment=experiment_name,
                   features=feature_description,model=experiment_name.split('_')[0],
                   params=result['model'].named_steps['model'].get_params(),cv=cv,
                   notes='StratifiedGroupKFold; same folds; no test inspection')
    else:
        print('로그 중복 방지:',experiment_name)
print('기본/파생피처 6개 실험 OOF/Test 저장 및 로그 확인 완료')


예측 저장: A_model__xgb_group_counts_1109_binary (CV 0.3459) → experiments/preds/
기록: log_A_model.csv ← xgb_group_counts_1109_binary (CV 0.3459)
예측 저장: A_model__lgbm_group_counts_1109_binary (CV 0.3263) → experiments/preds/
기록: log_A_model.csv ← lgbm_group_counts_1109_binary (CV 0.3263)
예측 저장: A_model__lr_group_counts_1109_binary (CV 0.2518) → experiments/preds/
기록: log_A_model.csv ← lr_group_counts_1109_binary (CV 0.2518)
예측 저장: A_model__xgb_group_counts_1109_derived (CV 0.4183) → experiments/preds/
기록: log_A_model.csv ← xgb_group_counts_1109_derived (CV 0.4183)
예측 저장: A_model__lgbm_group_counts_1109_derived (CV 0.3946) → experiments/preds/
기록: log_A_model.csv ← lgbm_group_counts_1109_derived (CV 0.3946)
예측 저장: A_model__lr_group_counts_1109_derived (CV 0.2475) → experiments/preds/
기록: log_A_model.csv ← lr_group_counts_1109_derived (CV 0.2475)
기본/파생피처 6개 실험 OOF/Test 저장 및 로그 확인 완료


### 기획서 지표 (Train OOF만 사용)

Top-1/Top-3는 OOF 기반입니다. 판단 보류 임계값은 아래에서 **탐색용**으로만 표시하며, 최종 성능 보고에는 별도 홀드아웃/내부 교차검증으로 임계값과 확률 보정을 검증해야 합니다.

In [40]:
from sklearn.metrics import accuracy_score, top_k_accuracy_score
rows = []
for name, item in RESULTS_1109.items():
    prob = item['cv']['oof_proba']
    top1 = accuracy_score(y, np.argmax(prob, axis=1))
    top3 = top_k_accuracy_score(y, prob, k=3, labels=np.arange(len(le.classes_)))
    confidence = np.max(prob, axis=1)
    for threshold in (0.30, 0.40, 0.50, 0.60):
        keep = confidence >= threshold
        rows.append({'model':name, 'threshold':threshold,
                     'top1_all':top1, 'top3_all':top3,
                     'coverage':float(keep.mean()),
                     'top1_accepted':float(accuracy_score(y[keep],np.argmax(prob[keep],axis=1))) if keep.any() else np.nan})
display(pd.DataFrame(rows))
print('주의: 보류 임계값은 동일 OOF에서 탐색한 값이므로 최종 확정 성능이 아닙니다.')

,model,threshold,top1_all,top3_all,coverage,top1_accepted
0,xgb_group_counts_1109,0.3,0.436220,0.695694,0.645864,0.559051
1,xgb_group_counts_1109,0.4,0.436220,0.695694,0.477665,0.640108
2,xgb_group_counts_1109,0.5,0.436220,0.695694,0.348170,0.714683
3,xgb_group_counts_1109,0.6,0.436220,0.695694,0.238832,0.806212
4,lgbm_group_counts_1109,0.3,0.420577,0.689889,0.866957,0.461682
5,lgbm_group_counts_1109,0.4,0.420577,0.689889,0.700532,0.519107
6,lgbm_group_counts_1109,0.5,0.420577,0.689889,0.552975,0.575386
7,lgbm_group_counts_1109,0.6,0.420577,0.689889,0.394936,0.643528
8,lr_group_counts_1109,0.3,0.269795,0.514756,0.939526,0.280639
9,lr_group_counts_1109,0.4,0.269795,0.514756,0.841155,0.297738


주의: 보류 임계값은 동일 OOF에서 탐색한 값이므로 최종 확정 성능이 아닙니다.


### 확률 보정·판단 보류 기준 (OOF 탐색 결과)

OOF 확률로 온도(temperature)를 교차 적합해 보정 전후 NLL/Brier/ECE를 비교합니다. 같은 데이터에서 학습된 기저 모델의 OOF 예측을 재사용하므로 **엄밀한 독립 검증/최종 확정 지표가 아닙니다**. 최종 기획서 수치에는 별도 홀드아웃 또는 nested CV가 필요합니다.


In [41]:
from scipy.special import softmax
from scipy.optimize import minimize_scalar
from sklearn.metrics import log_loss

def temperature_scale(p,t):
    logits=np.log(np.clip(np.asarray(p,dtype=float),1e-12,1))
    return softmax(logits/t,axis=1)

def multiclass_brier(y_true,p):
    onehot=np.eye(p.shape[1])[np.asarray(y_true,dtype=int)]
    return float(np.mean(np.sum((p-onehot)**2,axis=1)))

def ece_score(y_true,p,bins=10):
    conf=p.max(axis=1); correct=(p.argmax(axis=1)==y_true)
    edges=np.linspace(0,1,bins+1); total=0.0
    for i in range(bins):
        mask=(conf>=edges[i]) & (conf<(edges[i+1]) if i<bins-1 else conf<=edges[i+1])
        if mask.any(): total+=mask.mean()*abs(correct[mask].mean()-conf[mask].mean())
    return float(total)

calibration_rows=[]; calibrated_oof={}; abstain_rows=[]
for name,item in RESULTS_1109.items():
    p=np.asarray(item['cv']['oof_proba'],dtype=float)
    calibrated=np.empty_like(p)
    # calibration fold별: 해당 validation fold를 제외한 OOF로 T 적합
    for train_idx,val_idx in group_folds:
        fit_p=p[train_idx]; fit_y=np.asarray(y)[train_idx]
        objective=lambda t:log_loss(fit_y,temperature_scale(fit_p,t),labels=class_ids)
        opt=minimize_scalar(objective,bounds=(0.25,5.0),method='bounded')
        calibrated[val_idx]=temperature_scale(p[val_idx],float(opt.x))
    calibrated_oof[name]=calibrated
    for label,q in [('보정 전',p),('온도 보정 OOF(탐색)',calibrated)]:
        calibration_rows.append({'모델':name,'구분':label,'Top-1':float((q.argmax(axis=1)==y).mean()),
            'Top-3':float(np.mean([y[i] in np.argsort(q[i])[-3:] for i in range(len(y))])),
            'NLL':log_loss(y,q,labels=class_ids),'Brier':multiclass_brier(y,q),
            'ECE(10bins)':ece_score(y,q)})
    # 각 validation fold의 임계값을 다른 fold의 OOF에서 선택 (목표 수용 정확도 예시 0.70)
    confidence=calibrated.max(axis=1); predicted=calibrated.argmax(axis=1)
    for tr,va in group_folds:
        candidates=np.arange(0.2,0.91,0.05)
        feasible=[]
        for t in candidates:
            keep=confidence[tr]>=t
            if keep.sum()>=30 and np.mean(predicted[tr][keep]==np.asarray(y)[tr][keep])>=0.70:
                feasible.append(float(t))
        threshold=min(feasible) if feasible else 1.01  # 충족 못하면 전체 보류
        accept=confidence[va]>=threshold
        abstain_rows.append({'모델':name,'선택 임계값':threshold,'검증 Fold 인원':len(va),
            '수용 인원':int(accept.sum()),'수용 정답 수':int(np.sum(predicted[va][accept]==np.asarray(y)[va][accept]))})
display(pd.DataFrame(calibration_rows).round(4))
summary=pd.DataFrame(abstain_rows).groupby('모델',as_index=False)[['검증 Fold 인원','수용 인원','수용 정답 수']].sum()
summary['coverage']=summary['수용 인원']/summary['검증 Fold 인원']
summary['수용 정확도']=summary['수용 정답 수']/summary['수용 인원'].replace(0,np.nan)
display(summary.round(4))
print('주의: 목표 정확도 0.70은 예시 정책이며 임상적으로 검증된 기준이 아닙니다.')


c:\Users\user\Desktop\AI_Projects\ah07-cancer-subtype\.venv\Lib\site-packages\sklearn\metrics\_classification.py:304: UserWarning: The y_prob values do not sum to one. Make sure to pass probabilities.
  warnings.warn(
c:\Users\user\Desktop\AI_Projects\ah07-cancer-subtype\.venv\Lib\site-packages\sklearn\metrics\_classification.py:304: UserWarning: The y_prob values do not sum to one. Make sure to pass probabilities.
  warnings.warn(


,모델,구분,Top-1,Top-3,NLL,Brier,ECE(10bins)
0,xgb_group_counts_1109,보정 전,0.4362,0.6957,1.7925,0.6808,0.0184
1,xgb_group_counts_1109,온도 보정 OOF(탐색),0.4362,0.6957,1.7928,0.6808,0.0154
2,lgbm_group_counts_1109,보정 전,0.4206,0.6899,2.0446,0.7305,0.1529
3,lgbm_group_counts_1109,온도 보정 OOF(탐색),0.4206,0.6899,1.8709,0.6977,0.0193
4,lr_group_counts_1109,보정 전,0.2698,0.5148,3.7285,1.0528,0.3986
5,lr_group_counts_1109,온도 보정 OOF(탐색),0.2698,0.5148,2.5565,0.8633,0.0490


,모델,검증 Fold 인원,수용 인원,수용 정답 수,coverage,수용 정확도
0,lgbm_group_counts_1109,6201,1766,1302,0.2848,0.7373
1,lr_group_counts_1109,6201,15,8,0.0024,0.5333
2,xgb_group_counts_1109,6201,2117,1522,0.3414,0.7189


주의: 목표 정확도 0.70은 예시 정책이며 임상적으로 검증된 기준이 아닙니다.


### 암종별 근거 유전자 (로지스틱 회귀 계수, 탐색적 해석)

전체 Train에 적합한 로지스틱 회귀의 **유전자별 양의 계수 상위 10개**를 암종 이름과 함께 표시합니다. 계수는 인과관계나 임상적 바이오마커의 증거가 아닙니다. APC–대장암, VHL–신장암은 **해석 점검용 예시**이며 실제 피처 목록과 계수를 확인합니다.


In [42]:
# 최종 모델 선정 전 탐색적 해석. Train 전체로 학습한 모델이며 OOF 성능과 혼동하지 않습니다.
lr_key=next(k for k in RESULTS_1109 if k.startswith('lr_'))
lr_full=clone(RESULTS_1109[lr_key]['model']).fit(X,y)
feature_names=list(X.columns)+['n_mutated_genes','n_missense','n_synonymous','n_nonsense','n_frameshift','n_other']
coef=lr_full.named_steps['model'].coef_
assert coef.shape==(len(class_names),len(feature_names)),(coef.shape,len(feature_names))
importance=[]
for class_id, cancer in enumerate(class_names):
    weights=coef[class_id,:len(X.columns)]
    top=np.argsort(weights)[-10:][::-1]
    for rank,idx in enumerate(top,1):
        importance.append({'암종':cancer,'순위':rank,'유전자':feature_names[idx],
                           '표준화 계수':float(weights[idx])})
display(pd.DataFrame(importance).round(4))
for gene,cancers in [('APC',['COAD','READ','COADREAD']),('VHL',['KIRC','KIPAN'])]:
    matches=[i for i,col in enumerate(X.columns) if str(col).upper()==gene]
    if not matches:
        print(f'{gene}: 입력 유전자 목록에 없음')
        continue
    idx=matches[0]
    print(f'\n{gene} 유전자 계수 (참고용)')
    display(pd.DataFrame({'암종':class_names,'계수':coef[:,idx]}).query('암종 in @cancers').round(4))


,암종,순위,유전자,표준화 계수
0,ACC,1,SOWAHC,0.3074
1,ACC,2,PTGIR,0.3021
2,ACC,3,PLEC,0.2938
3,ACC,4,CMPK2,0.2572
4,ACC,5,CKB,0.2271
...,...,...,...,...
255,UCEC,6,CTCF,0.3453
256,UCEC,7,CDKN1B,0.2902
257,UCEC,8,TP53,0.2605
258,UCEC,9,PANX1,0.2581



APC 유전자 계수 (참고용)


,암종,계수
4,COAD,1.2605



VHL 유전자 계수 (참고용)


,암종,계수
8,KIPAN,1.3489
9,KIRC,1.4734
